# Week 4 — Frequency & KPI (2024)

In this notebook we integrate the official dataset of **train frequency (dispatched trains)** with the demand dataset from turnstiles.
The goal is to calculate a **key KPI**: passengers per train dispatched, as a proxy for efficiency.

**Datasets used:**
- `data/raw/frecuencia/frecuencia_subte.xlsx` (train frequency by line and date)
- Demand dataset processed in Week 2 (`mol_full`, trend_full)

**Outputs:**
- Aggregated monthly frequency by line
- KPI: passengers per train
- CSV and PNG exports in `/data/processed/` and `/assets/screenshots/`


In [2]:
# === Block 1: Load & normalize wide frequency sheet ===
import os, re
import pandas as pd

BASE_DIR   = os.path.abspath("..")
RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")
FREQ_PATH  = os.path.join(RAW_DIR, "frecuencia", "frecuencia_subte.xlsx")

# 1) Abrir Excel (única hoja: 'Frecuencias')
xls = pd.ExcelFile(FREQ_PATH)
print("Sheets available:", xls.sheet_names)

freq_raw = pd.read_excel(FREQ_PATH, sheet_name=0)
print("Raw shape:", freq_raw.shape)
print("Columns:", freq_raw.columns.tolist())

# 2) Normalizar columnas
freq_raw.columns = [c.strip().lower() for c in freq_raw.columns]

# Esperamos: 'mes_anio' + 'servicio_frecuencia_a' ... 'servicio_frecuencia_h' (y quizá premetro)
date_col = "mes_anio"
wide_cols = [c for c in freq_raw.columns if c.startswith("servicio_frecuencia_")]

# Filtrar sólo líneas de Subte (A, B, C, D, E, H)
valid_suffix = {"a","b","c","d","e","h"}
wide_cols = [c for c in wide_cols if c.split("servicio_frecuencia_")[-1] in valid_suffix]

assert date_col in freq_raw.columns, "No se encontró la columna 'mes_anio' en el Excel."
assert wide_cols, "No se detectaron columnas 'servicio_frecuencia_*' para A/B/C/D/E/H."

# 3) Pasar a formato long (tidy)
freq_long = freq_raw.melt(
    id_vars=[date_col],
    value_vars=wide_cols,
    var_name="service",
    value_name="dispatched_trains"
)

# 4) Mapear línea desde 'service'
freq_long["line"] = (
    freq_long["service"].str.replace("servicio_frecuencia_", "", regex=False).str.upper()
)

# 5) Parsear mes (mes_anio) a 'year_month' (YYYY-MM)
# Intento directo
freq_long["date"] = pd.to_datetime(freq_long[date_col], errors="coerce")

# Si viniera como texto tipo "ene-24" o "ene/2024", probamos un parser alternativo
if freq_long["date"].isna().mean() > 0:
    # Reemplazos comunes de meses en español -> inglés abreviado (por si hacen falta)
    month_map = {
        "ene":"jan","feb":"feb","mar":"mar","abr":"apr","may":"may","jun":"jun",
        "jul":"jul","ago":"aug","sep":"sep","oct":"oct","nov":"nov","dic":"dec"
    }
    tmp = freq_long[date_col].astype(str).str.lower()
    # normaliza separadores
    tmp = tmp.str.replace(r"[./]", "-", regex=True)
    # reemplaza meses
    for es,en in month_map.items():
        tmp = tmp.str.replace(fr"\b{es}\b", en, regex=True)
    freq_long["date"] = pd.to_datetime(tmp, errors="coerce")

# Fallback: si aún hay NaT, fuerza como periodo agregando día 1 cuando sea numérico tipo '2024-01'
mask_nat = freq_long["date"].isna()
if mask_nat.any():
    # intenta extraer YYYY-MM con regex
    ym = freq_long.loc[mask_nat, date_col].astype(str).str.extract(r"(\d{4})[-/](\d{1,2})")
    if not ym.isna().all().all():
        ym = ym.fillna(method="ffill")  # por si hubiera filas vacías
        freq_long.loc[mask_nat, "date"] = pd.to_datetime(
            ym[0] + "-" + ym[1].str.zfill(2) + "-01", errors="coerce"
        )

# 6) Limpieza final
freq_long["year_month"] = freq_long["date"].dt.to_period("M").astype(str)
freq_long["dispatched_trains"] = pd.to_numeric(freq_long["dispatched_trains"], errors="coerce").fillna(0).astype("int64")
freq_long = freq_long.drop(columns=["service"])  # ya extraímos 'line'
freq_long = freq_long[["year_month", "line", "dispatched_trains"]].sort_values(["year_month","line"]).reset_index(drop=True)

print("Normalized shape:", freq_long.shape)
display(freq_long.head(12))
print("Year-Month range:", freq_long["year_month"].min(), "→", freq_long["year_month"].max())
print(freq_long.groupby("line")["dispatched_trains"].sum().rename("total_trains").reset_index())


Sheets available: ['Frecuencias']
Raw shape: (83, 8)
Columns: ['mes_anio', 'servicio_frecuencia_a', 'servicio_frecuencia_b', 'servicio_frecuencia_c', 'servicio_frecuencia_d', 'servicio_frecuencia_e', 'servicio_frecuencia_h', 'servicio_frecuencia_premetro']
Normalized shape: (498, 3)


,year_month,line,dispatched_trains
0,2019-01,A,0
1,2019-01,B,0
2,2019-01,C,0
3,2019-01,D,0
4,2019-01,E,0
5,2019-01,H,0
6,2019-02,A,0
7,2019-02,B,0
8,2019-02,C,0
9,2019-02,D,0


Year-Month range: 2019-01 → 2025-11
  line  total_trains
0    A             0
1    B             0
2    C             0
3    D             0
4    E             0
5    H             0


In [3]:
# === Block 1.1: Diagnose & Fix numeric parsing ===
# 1) Muestra rápida de los primeros valores "en crudo" (después del melt)
print("Sample raw values by line (first 10 rows each):")
for ln in ["A","B","C","D","E","H"]:
    sample = (freq_long.query("line == @ln")
              .head(10)["dispatched_trains"]
              .astype(str)
              .tolist())
    print(f"Line {ln}:", sample)

# 2) Conteo de no-nulos antes de limpiar
nn_before = freq_long["dispatched_trains"].notna().sum()
print("\nNon-null count BEFORE cleaning:", nn_before)

# 3) Limpieza: quedarnos sólo con dígitos
freq_long["raw_value"] = freq_long["dispatched_trains"].astype(str)
freq_long["dispatched_trains"] = pd.to_numeric(
    freq_long["raw_value"].str.replace(r"[^0-9]", "", regex=True),
    errors="coerce"
)

# 4) Rellenar NaN con 0 y castear a int
freq_long["dispatched_trains"] = freq_long["dispatched_trains"].fillna(0).astype("int64")

# 5) Verificación rápida tras limpieza
nn_after = freq_long["dispatched_trains"].ne(0).sum()
print("Non-zero count AFTER cleaning:", nn_after)

print("\nTotals by line after cleaning:")
display(freq_long.groupby("line")["dispatched_trains"].sum().rename("total_trains").reset_index())

print("Year-Month span:", freq_long["year_month"].min(), "→", freq_long["year_month"].max())
display(freq_long.sort_values(["year_month","line"]).head(12))


Sample raw values by line (first 10 rows each):
Line A: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Line B: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Line C: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Line D: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Line E: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']
Line H: ['0', '0', '0', '0', '0', '0', '0', '0', '0', '0']

Non-null count BEFORE cleaning: 498
Non-zero count AFTER cleaning: 0

Totals by line after cleaning:


,line,total_trains
0,A,0
1,B,0
2,C,0
3,D,0
4,E,0
5,H,0


Year-Month span: 2019-01 → 2025-11


,year_month,line,dispatched_trains,raw_value
0,2019-01,A,0,0
1,2019-01,B,0,0
2,2019-01,C,0,0
3,2019-01,D,0,0
4,2019-01,E,0,0
5,2019-01,H,0,0
6,2019-02,A,0,0
7,2019-02,B,0,0
8,2019-02,C,0,0
9,2019-02,D,0,0


In [1]:
# === Block 2A: Derive dispatched trains from "formaciones-despachadas-2024.xlsx" ===
import os
import pandas as pd

BASE_DIR = os.path.abspath("..")  # desde /notebooks
RAW_DIR  = os.path.join(BASE_DIR, "data", "raw")
FORM_XLS = os.path.join(RAW_DIR, "formaciones", "formaciones-despachadas-2024.xlsx")

# Carga robusta (detect sheet)
xls = pd.ExcelFile(FORM_XLS)
print("Sheets in formaciones-despachadas:", xls.sheet_names)

# Intentar hoja obvia o la primera
pref_sheets = [s for s in xls.sheet_names if "formaciones" in s.lower() or "desp" in s.lower()]
sheet_use = pref_sheets[0] if pref_sheets else xls.sheet_names[0]
form_raw = pd.read_excel(FORM_XLS, sheet_name=sheet_use)
print("Raw shape:", form_raw.shape)
print("Columns:", form_raw.columns.tolist())

# Normalizamos nombres
form_raw.columns = [c.strip().lower() for c in form_raw.columns]

# Heurísticas comunes
# buscamos fecha, línea y alguna métrica de conteo de formaciones/servicios
date_col = next((c for c in form_raw.columns if c in ["fecha","date","dia","día"]), None)
line_col = next((c for c in form_raw.columns if c in ["linea","línea","line"]), None)

# si viene en wide (una col por línea), la derretimos
wide_line_cols = [c for c in form_raw.columns if c.startswith("linea") or c in list("abcdeh")]
if date_col and wide_line_cols and (line_col is None):
    tmp = form_raw[[date_col] + wide_line_cols].copy()
    long = tmp.melt(id_vars=[date_col], var_name="line", value_name="dispatched_trains")
    # normalizar line → "A/B/C/D/E/H"
    long["line"] = (long["line"].astype(str)
                    .str.upper()
                    .str.replace("LINEA", "", regex=False)
                    .str.strip())
else:
    # formato largo ya con columna de línea
    # intentar detectar una col de conteo
    cnt_cands = [c for c in form_raw.columns if "forma" in c or "despach" in c or "serv" in c or "tren" in c]
    cnt_col = cnt_cands[0] if cnt_cands else None
    assert date_col and line_col and cnt_col, "No pude detectar columnas de fecha/línea/conteo en formaciones."
    long = form_raw[[date_col, line_col, cnt_col]].copy()
    long = long.rename(columns={date_col:"date", line_col:"line", cnt_col:"dispatched_trains"})

# Tipos y limpieza
if "date" not in long.columns:
    long = long.rename(columns={date_col:"date"})
long["date"] = pd.to_datetime(long["date"], errors="coerce", dayfirst=True)
long = long.dropna(subset=["date"])
long["line"] = (long["line"].astype(str).str.upper()
                .str.replace(r"^LINEA\s*", "", regex=True).str.strip())
long["dispatched_trains"] = pd.to_numeric(long["dispatched_trains"], errors="coerce").fillna(0).astype("int64")

# Agregado mensual por línea
long["year_month"] = long["date"].dt.to_period("M").astype(str)
freq_from_form = (long.groupby(["year_month","line"], as_index=False)["dispatched_trains"]
                       .sum()
                       .sort_values(["year_month","line"]))

print("freq_from_form shape:", freq_from_form.shape)
display(freq_from_form.head(12))

print("Totals per line:")
display(freq_from_form.groupby("line")["dispatched_trains"].sum().rename("total_trains").reset_index())


Sheets in formaciones-despachadas: ['formaciones-despachadas-2024']
Raw shape: (495719, 20)
Columns: ['Fecha', 'Linea', 'Tipo ', 'Regist', 'Orden', 'Tren', 'Nombre  Formación', 'Modelo  Formación', 'Causa A', 'Descripción A', 'Causa D', 'Descripción D', 'Cant Coches A', 'Cant Coches D', 'Km A', 'Km D', 'Tipo Viaje A', 'Tipo Viaje D', 'Hora Sale A', 'Hora Sale D']
freq_from_form shape: (83, 3)


,year_month,line,dispatched_trains
0,2024-01,A,51972
1,2024-01,B,42821
2,2024-01,C,31863
3,2024-01,D,8878
4,2024-01,E,20056
5,2024-01,H,27027
6,2024-01,P,10453
7,2024-02,A,49585
8,2024-02,B,38332
9,2024-02,C,30485


Totals per line:


,line,total_trains
0,A,781209
1,B,605261
2,C,473934
3,D,375795
4,E,438603
5,H,500891
6,P,124602


In [4]:
# === Week 4 — Block 3A (FINAL): Robust fallback + KPI build & export ===
import os, re, glob, csv
from collections import defaultdict
import pandas as pd
import plotly.express as px

# ---- Paths (self-contained) ----
BASE_DIR   = os.path.abspath("..")  # desde /notebooks
RAW_DIR    = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR   = os.path.join(BASE_DIR, "data", "processed")
ASSETS_DIR = os.path.join(BASE_DIR, "assets", "screenshots")
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)

# ---- 1) Load frequency (from 'formaciones-despachadas-2024.xlsx') ----
freq_path = os.path.join(PROC_DIR, "freq_from_form_2024.csv")
assert os.path.exists(freq_path), "freq_from_form_2024.csv no existe. Ejecutá el bloque 3 primero."
freq = pd.read_csv(freq_path, dtype={"year_month": str, "line": str, "dispatched_trains": "Int64"})
freq["line"] = freq["line"].str.upper().str.strip()
freq = freq.loc[freq["line"].isin(list("ABCDEFHP"))].copy()  # nos quedamos con A–H (+P si aparece)
freq = freq[["year_month","line","dispatched_trains"]]

# ---- 2) Try to load passengers trend (fast path) ----
trend_path = os.path.join(PROC_DIR, "trend_full_2024.csv")
if os.path.exists(trend_path):
    trend_full = pd.read_csv(trend_path, dtype={"year_month": str, "line": str, "passengers": "Int64"})
    print("Loaded trend_full from:", trend_path, "→", trend_full.shape)
else:
    print("trend_full CSV not found → fallback: rebuild from raw molinetes (robust reader)")
    # ---------- Robust reader (multi-encoding + unique headers) ----------
    def normalize_token(s: str) -> str:
        s = (s or "").strip().strip('"').strip()
        s = re.sub(r"\s+", "_", s)
        return s.lower() or "col"

    def make_unique(cols):
        seen = defaultdict(int)
        out = []
        for c in cols:
            base = normalize_token(c)
            seen[base] += 1
            out.append(base if seen[base] == 1 else f"{base}_{seen[base]-1}")
        return out

    def read_header_names_unique_multi(path, seps=(";",), encodings=("utf-8-sig","latin1","cp1252")):
        # intenta varias combinaciones hasta encontrar separador/encoding que dé >1 columna
        for enc in encodings:
            try:
                with open(path, "r", encoding=enc, errors="strict") as f:
                    first_line = f.readline().rstrip("\n\r")
                for sep in seps:
                    raw_cols = first_line
                    if raw_cols.startswith('"') and raw_cols.endswith('"'):
                        raw_cols = raw_cols[1:-1]
                    cols = [c for c in raw_cols.split(sep)]
                    if len(cols) > 1:
                        return make_unique(cols), enc, sep
            except Exception:
                continue
        # último recurso permisivo
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            first_line = f.readline().rstrip("\n\r")
        sep = ";"
        if first_line.startswith('"') and first_line.endswith('"'):
            first_line = first_line[1:-1]
        cols = [c for c in first_line.split(sep)]
        return make_unique(cols), "utf-8", sep

    def read_molinetes_unique_multi(path):
        cols, enc, sep = read_header_names_unique_multi(path)
        # probar varios encodings reales al leer
        for enc_try in (enc, "utf-8-sig", "latin1", "cp1252"):
            try:
                df = pd.read_csv(
                    path, sep=sep, encoding=enc_try, engine="python",
                    header=None, names=cols, quoting=csv.QUOTE_NONE, on_bad_lines="skip"
                )
                for c in df.select_dtypes(include="object").columns:
                    df[c] = df[c].astype(str).str.strip('"').str.strip()
                return df
            except Exception:
                continue
        raise ValueError(f"No se pudo leer {os.path.basename(path)} con encodings comunes")

    MOL_DIR = os.path.join(RAW_DIR, "molinetes")
    csvs = sorted(glob.glob(os.path.join(MOL_DIR, "*.csv")))
    print("Reading molinetes CSVs:", len(csvs))
    ok, bad = 0, 0
    df_list = []
    for p in csvs:
        try:
            df_list.append(read_molinetes_unique_multi(p))
            ok += 1
        except Exception as e:
            print("FAIL:", os.path.basename(p), "→", e)
            bad += 1
    print(f"OK files: {ok} | Failed: {bad}")
    assert ok > 0, "No se pudo leer ningún CSV de molinetes."

    mol = pd.concat(df_list, ignore_index=True)

    # Canonicalizar nombres usuales
    mol.columns = [c.strip().lower() for c in mol.columns]
    rename_map = {}
    for c in mol.columns:
        if c in {"fecha"}: rename_map[c] = "date"
        if c in {"desde", "desde_hora", "hora_desde"}: rename_map[c] = "time_from"
        if c in {"hasta", "hasta_hora", "hora_hasta"}: rename_map[c] = "time_to"
        if c in {"linea", "línea", "line"}: rename_map[c] = "line"
        if c in {"estacion", "estación", "station"}: rename_map[c] = "station"
        if c in {"pax_total","viajes","pasajeros","pax","passengers","conteo","count"}:
            rename_map[c] = "passengers"
    mol = mol.rename(columns=rename_map)

    # Filtrar filas tipo header coladas
    mask_header_row = (
        mol.get("time_from", "").astype(str).str.upper().eq("DESDE") |
        mol.get("time_to", "").astype(str).str.upper().eq("HASTA")
    )
    mol = mol.loc[~mask_header_row].copy()

    # Tipos/normalización
    mol["date"] = pd.to_datetime(mol.get("date"), errors="coerce", dayfirst=True)
    mol = mol.loc[mol["date"].notna()].copy()
    mol["year_month"] = mol["date"].dt.to_period("M").astype(str)
    for col in ["station","time_from","time_to"]:
        if col in mol.columns:
            mol[col] = mol[col].astype(str).str.strip().str.upper()
    if "line" in mol.columns:
        mol["line"] = (mol["line"].astype(str).str.upper()
                       .str.replace(r"^LINEA\s*", "", regex=True)
                       .str.strip())
    mol["passengers"] = pd.to_numeric(mol.get("passengers"), errors="coerce").fillna(0).astype("int64")

    # Agregado mensual por línea (2024)
    trend_full = (
        mol.loc[mol["date"].dt.year.eq(2024)]
           .groupby(["year_month","line"], as_index=False)["passengers"].sum()
           .sort_values(["year_month","line"])
    )
    trend_full.to_csv(trend_path, index=False, encoding="utf-8")
    print("Rebuilt & saved trend_full →", trend_path, "shape:", trend_full.shape)

# ---- 3) KPI: passengers per dispatched train ----
# Mantener solo líneas A–H y 2024
trend_2024 = trend_full.loc[trend_full["year_month"].str.startswith("2024-")].copy()
trend_2024["line"] = trend_2024["line"].str.upper().str.strip()
freq_2024 = freq.loc[freq["year_month"].str.startswith("2024-")].copy()

kpi = pd.merge(trend_2024, freq_2024, on=["year_month","line"], how="inner")
kpi["kpi_pax_per_train"] = (kpi["passengers"] / kpi["dispatched_trains"].replace(0, pd.NA)).round(2)

# Snapshot 2024 (suma por línea)
kpi_2024 = (
    kpi.groupby("line", as_index=False)
       .agg(passengers=("passengers","sum"),
            dispatched_trains=("dispatched_trains","sum"))
)
kpi_2024["kpi_pax_per_train"] = (kpi_2024["passengers"] / kpi_2024["dispatched_trains"].replace(0, pd.NA)).round(2)

# ---- 4) Export CSVs ----
kpi_2024_path = os.path.join(PROC_DIR, "kpi_pax_per_train_2024_by_line.csv")
kpi_trend_path = os.path.join(PROC_DIR, "kpi_pax_per_train_2024_trend.csv")
kpi_2024.to_csv(kpi_2024_path, index=False, encoding="utf-8")
kpi.to_csv(kpi_trend_path, index=False, encoding="utf-8")
print("Saved CSV:", kpi_2024_path, "|", kpi_trend_path)

# ---- 5) Charts (Kaleido) ----
fig_bar = px.bar(
    kpi_2024.sort_values("kpi_pax_per_train", ascending=False),
    x="line", y="kpi_pax_per_train", text="kpi_pax_per_train",
    title="KPI — Passengers per Dispatched Train (2024)",
    labels={"line":"Line", "kpi_pax_per_train":"Pax / Train"}
)
fig_bar.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_bar.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')

fig_line = px.line(
    kpi.sort_values(["year_month","line"]),
    x="year_month", y="kpi_pax_per_train", color="line", markers=True,
    title="Monthly KPI — Pax / Train by Line (2024)",
    labels={"year_month":"Year-Month", "kpi_pax_per_train":"Pax / Train"}
)
fig_line.update_layout(xaxis_tickangle=-45)

out_bar  = os.path.join(PROC_DIR, "kpi_pax_per_train_2024_by_line.png")
out_line = os.path.join(PROC_DIR, "kpi_pax_per_train_2024_trend.png")
fig_bar.write_image(out_bar,  scale=2)
fig_line.write_image(out_line, scale=2)
print("Saved charts:", out_bar, "|", out_line)


trend_full CSV not found → fallback: rebuild from raw molinetes (robust reader)
Reading molinetes CSVs: 24
OK files: 24 | Failed: 0
Rebuilt & saved trend_full → c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\trend_full_2024.csv shape: (72, 3)
Saved CSV: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\kpi_pax_per_train_2024_by_line.csv | c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\kpi_pax_per_train_2024_trend.csv
Saved charts: c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\kpi_pax_per_train_2024_by_line.png | c:\Users\do_ch\OneDrive\Escritorio\Proyectos\Proyectos GitHub\subte-dashboard\data\processed\kpi_pax_per_train_2024_trend.png
